# **Cadillac Tyre Degradation Analysis - Japan Race 2026**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import fastf1

#### Loading Japan Race 2026 data:

In [ ]:
session = fastf1.get_session(2026, 'Japan', 'R')
session.event

In [ ]:
session.load()
session.results

#### Retrieving lap data of Bottas, Perez and Competitors (Hulkendberg, Bortoleto):

In [ ]:
drivers = session.laps.pick_drivers(['BOT', 'PER', 'HUL', 'BOR'])

## **Visualising all drivers stints**

In [ ]:
driver_stints = (drivers
                 .groupby(['Driver', 'Stint', 'Compound'])
                 .size()
                 .reset_index(name='LapCount'))

stints_fig, stints_ax = plt.subplots()

compound_colours = {
    'HARD' : 'White',
    'MEDIUM' : 'Yellow',
    'SOFT' : 'Red',
    'INTERMEDIATE' : 'Green',
    'WET' : 'Blue'
}

legend_elements=[
    Patch(facecolor='White', edgecolor='Black', label="HARD"),
    Patch(facecolor='Yellow', edgecolor='Black', label="MEDIUM"),
    Patch(facecolor='Red', edgecolor='Black', label="SOFT"),
    Patch(facecolor='Green', edgecolor='Black', label="INTERMEDIATE"),
    Patch(facecolor='Blue', edgecolor='Black', label="WET")
]

for driver, group in driver_stints.groupby('Driver'):
    left = 0

    for _, row in group.iterrows():
        stints_ax.barh(
            driver,
            row['LapCount'],
            left=left,
            color=compound_colours.get(row['Compound']),
            edgecolor='Black',
        )
        left += row['LapCount']
    
    

plt.xlabel('Laps')
plt.ylabel('Drivers')
plt.title('Driver Stints')
plt.legend(handles=legend_elements, title='Compounds', bbox_to_anchor=(1.05, 1), loc='best')
plt.show()

## **Laptimes of Selected Drivers When Using Hard Compounds Compared to Grid Average**

#### Calculating Lap Time Averages of All Drivers and Selected Drivers on Hard Compounds

In [ ]:
# Calculating the average grid-wide lap time of all laps with hard compound excluding in-laps, out-laps, yellow flags, safety cars and the first lap (0 days 00:01:41.094228813 average when you include all laps with hard compound)
all_drivers_on_hard = session.laps[
    (session.laps['Compound'] == 'HARD') & 
    (session.laps['PitInTime'].isna()) & 
    (session.laps['PitOutTime'].isna()) & 
    (session.laps['TrackStatus'] == '1') &
    (session.laps['LapNumber'] > 1)
]
lap_average_on_hard = all_drivers_on_hard['LapTime'].mean()

selected_drivers_on_hard = drivers[
    (drivers['Compound'] == 'HARD') &
    (session.laps['PitInTime'].isna()) & 
    (session.laps['PitOutTime'].isna()) & 
    (session.laps['TrackStatus'] == '1') &
    (session.laps['LapNumber'] > 1)
]
selected_drivers_lap_average_on_hard = selected_drivers_on_hard['LapTime'].mean()

print("The average lap time for all drivers on hard compound: ", lap_average_on_hard)
print("The average lap time of selected drivers on hard compound: ", selected_drivers_lap_average_on_hard)